# Day 7 — Citation coverage + refusal gating

**Goal:** enforce that generated drafts include citations for each bullet, and refuse when retrieval confidence is low or citations are missing.

In [ ]:
!pip -q install pandas numpy faiss-cpu

In [ ]:

# === Setup ===
from google.colab import drive
drive.mount('/content/drive')

import os, re, json, math, random, time
import numpy as np
import pandas as pd
from tqdm import tqdm

BASE = "/content/drive/MyDrive/mimic_project"
RAW  = f"{BASE}/raw"
DATA = f"{BASE}/dataset"
IMAGES = f"{DATA}/images"
REPORTS_DIR = f"{DATA}/reports"

os.makedirs(RAW, exist_ok=True)
os.makedirs(IMAGES, exist_ok=True)
os.makedirs(REPORTS_DIR, exist_ok=True)

print("BASE:", BASE)
print("RAW:", RAW)
print("IMAGES:", IMAGES)
print("REPORTS_DIR:", REPORTS_DIR)


## 1) Load data + index

In [ ]:

import faiss

df = pd.read_csv(f"{BASE}/clean_dataset_2696.csv")
fused = np.load(f"{BASE}/fused_embeddings_alpha_0.5.npy").astype("float32")
fused = fused / np.linalg.norm(fused, axis=1, keepdims=True)

index = faiss.IndexFlatIP(fused.shape[1])
index.add(fused)


## 2) Citation checker + strict RAG runner

In [ ]:

def retrieve(query_idx: int, k=3):
    q = fused[query_idx:query_idx+1]
    scores, inds = index.search(q, k+1)
    inds = [i for i in inds[0].tolist() if i != query_idx][:k]
    cases = []
    for ci, i in enumerate(inds, start=1):
        cases.append({
            "case": ci,
            "row_idx": i,
            "study_id": int(df.loc[i,"study_id"]),
            "score": float(scores[0][ci]),
            "impression": df.loc[i,"impression"]
        })
    best = float(scores[0][1]) if len(scores[0]) > 1 else 0.0
    return cases, best

def draft_from_cases(cases, max_bullets=2):
    bullets = []
    for c in cases:
        s = c["impression"].replace("\n"," ").strip()
        first = re.split(r"(?<=[.!?])\s+", s)[0].strip()
        if first:
            bullets.append(f"- {first} [Case {c['case']}]")
        if len(bullets) >= max_bullets:
            break
    return "IMPRESSION:\n" + ("\n".join(bullets) if bullets else "- Unable to draft.")

def citation_coverage(draft: str, k=3):
    needed = [f"[Case {i}]" for i in range(1, k+1)]
    present = [c for c in needed if c in draft]
    missing = [c for c in needed if c not in draft]
    cov = len(present) / max(1, len(needed))
    return {"needed": needed, "present": present, "missing": missing, "coverage": cov}

def run_strict(query_idx: int, k=3, min_score=0.90):
    cases, best = retrieve(query_idx, k=k)
    draft = draft_from_cases(cases, max_bullets=2)
    check = citation_coverage(draft, k=k)

    if best < min_score:
        return {
            "status": "refused",
            "best_score": best,
            "coverage": check["coverage"],
            "draft": "IMPRESSION:\n- Unable to generate reliable draft due to low retrieval confidence.",
            "cases": cases,
            "check": check
        }

    if check["coverage"] < 1.0:
        return {
            "status": "refused",
            "best_score": best,
            "coverage": check["coverage"],
            "draft": "IMPRESSION:\n- Unable to generate reliable draft due to missing citations.",
            "cases": cases,
            "check": check
        }

    return {
        "status": "generated",
        "best_score": best,
        "coverage": check["coverage"],
        "draft": draft,
        "cases": cases,
        "check": check
    }

out = run_strict(query_idx=0, k=3, min_score=0.90)
print("status:", out["status"])
print("best_score:", out["best_score"])
print("coverage:", out["coverage"])
print(out["draft"])


## 3) Batch test (n=10) and log results

In [ ]:

n = 10
rows = []
for qi in range(n):
    out = run_strict(qi, k=3, min_score=0.90)
    rows.append({
        "query_study_id": int(df.loc[qi,"study_id"]),
        "status": out["status"],
        "best_score": out["best_score"],
        "citation_coverage": out["coverage"],
        "missing_citations": ",".join(out["check"]["missing"])
    })

log_df = pd.DataFrame(rows)
print(log_df)

out_path = f"{BASE}/rag_outputs_day7.csv"
log_df.to_csv(out_path, index=False)
print("Saved log to:", out_path, "Rows:", len(log_df))


✅ **Day 7 deliverables:** strict RAG runner with confidence gating + citation coverage logs.